## Workspace setup

In [ ]:
from datetime import datetime  
import uproot
from functools import partial
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import importlib

# Notebooks run from VSCode use home directory as a base path
# while notebooks run from JupyterLab use the current directory as a base path
import sys
sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")

# Change directory to the working directory
import os
os.chdir('/scratch_hdd/akalinow/ELITPC/PythonAnalysis/')

## Training dataset preparation

In [ ]:
import io_functions as io
importlib.reload(io)

import plotting_functions as plf
importlib.reload(plf)

import utility_functions as utils
importlib.reload(utils)

batchSize = 32
dataset = tf.data.Dataset.load('SimEvent_Track3D_TwoProng_gun_MC', compression="GZIP")
dataset = dataset.batch(batchSize, drop_remainder=True)
dataset = dataset.map(lambda x,y: (tf.reshape(x, (-1,)+io.projections.shape), tf.reshape(y, (-1,3,3))))
dataset = dataset.map(lambda x,y: (x, utils.XYZtoUVWT_event(y)))
dataset = dataset.map(lambda x,y: (x, tf.reshape(y, (-1,12))))
dataset = dataset.map(lambda x,y: (x, tf.gather(y, batch_dims=1,  indices = [[0,1,2,3]]*32))) #select only first 4 features - vtx coordinates
dataset = dataset.map(lambda x,y: (tf.where(x>0.1, x, 0), y))
dataset = dataset.prefetch(tf.data.AUTOTUNE)


## Model definition

In [ ]:
model = tf.keras.Sequential([
  tf.keras.layers.Input(shape=(256,512,3), name="input_image", dtype=tf.float32),
  tf.keras.layers.Conv2D(3*16, kernel_size=(8,8), groups=3, padding='same', activation='relu', 
                         data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
  tf.keras.layers.Conv2D(3*8, kernel_size=(4,4), groups=3, padding='same', activation='relu', 
                         data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
  tf.keras.layers.Conv2D(3*4, kernel_size=(2,2), groups=3, padding='same', activation='relu', 
                         data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dense(32, activation='relu'), 
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dense(4, activation="sigmoid"),
  tf.keras.layers.Rescaling(512)
])

decay_steps = dataset.cardinality().numpy() # decay step every epoch
initial_learning_rate = 0.001
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                decay_steps=decay_steps,
                decay_rate=0.98,
                staircase=False)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
model.compile(optimizer = optimizer, 
              loss = 'mse', 
              metrics=['mse']) 

model.summary()
model.evaluate(dataset.take(10))

## Model training

In [ ]:
%%time

import plotting_functions as plf
importlib.reload(plf)

log_dir = "logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
#tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, profile_batch=(10, 20))
early_stop_callback = tf.keras.callbacks.EarlyStopping(patience=2, verbose=1)
callbacks =  [early_stop_callback]

epochs=100
history = model.fit(dataset,
                    epochs=epochs,
                    verbose = 1,
                    validation_data = dataset.take(10),
                    #callbacks=callbacks
                    )
plf.plotTrainHistory(history)

current_time = datetime.now().strftime("%Y_%b_%d_%H_%M_%S")
print("Training start. Current Time =", current_time)
job_dir = f"training/{epochs:04d}_"+current_time+".keras"
print("Saving model to", job_dir)
model.save(job_dir)

## Model performance on training data.

Fill Pandas DataFrame with true and response values.

In [ ]:
%%time
import utility_functions as utils
importlib.reload(utils)

#model_path = "./training/2023_Apr_28_16_58_32/"
#model = tf.keras.models.load_model(model_path)

columnsUVWT = utils.columnsUVWT[[0,1,2,3]] #vertex position only
#columnsUVWT = utils.columnsUVWT

df = utils.getEmptyPandasDataset(columnsUVWT)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

for aBatch in dataset: 
    df = utils.fillPandasDataset(aBatch, df, model)     
    
df.describe()    

### Resolution plots

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

#plf.controlPlots(df)
plf.plotEndPointRes(df=df, edge="Vtx", coordinates=["u", "v", "w", "t"])
#plf.plotEndPointRes(df=df, edge="Alpha", coordinates=["u", "v", "w", "t"])
#plf.plotEndPointRes(df=df, edge="Carbon", coordinates=["u", "v", "w", "t"])

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

#path = '/home/akalinow/scratch/ELITPC/PythonAnalysis/training/0005_2025_Jul_16_15_00_08.keras'
#model = tf.keras.models.load_model(path)

x = iter(dataset)
for i in range(15):
    item = next(x)
    plf.plotEvent(item, model=model)